# 02 - LangGraph Workflow：建立多步驟 LLM 工作流

本筆記本示範如何直接使用 LangGraph `StateGraph` API 建構 workflow，
並透過 `call_llm` node 搭配 MLflow span 追蹤 LLM 呼叫細節。

## 核心概念

本框架直接採用 LangGraph 原生架構，不再包裝額外抽象層：

- **`state.py`**：`BaseState` 繼承 LangGraph `MessagesState`，只有 messages
- **`nodes.py`**：提供通用 node functions（如 `create_call_llm_node`）
- **`build_workflow.py`**：存放預建的 workflow 範例

你可以直接使用 LangGraph 的 `StateGraph`、`add_node`、`add_edge`、`set_entry_point` 等 API。
自行擴展 `BaseState` 加入所需欄位（如 `llm_kwargs`、`image_base64` 等）。

In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath(".."))

from app.utils.config import init_config
from app.logger import init_mlflow
from app.workflow import BaseState, create_call_llm_node
from langgraph.graph import END, StateGraph

cfg = init_config()
init_mlflow(cfg)
print("設定與 MLflow 初始化完成。")

## BaseState 結構

`BaseState` 繼承自 LangGraph `MessagesState`，只有 `messages` 欄位。

需要額外欄位時，自行擴展：

```python
from app.workflow.state import BaseState

class MyState(BaseState):
    image_base64: str
    llm_kwargs: dict[str, Any]
```

## 範例一：使用預建 workflow

`build_simple_chain()` 建構最簡單的 LLM chain：user message → call_llm → END

In [ ]:
from app.workflow.build_workflow import build_simple_chain

# 使用預建的 simple chain（自動從 llm_config.yaml 取得 ChatLiteLLM）
graph = build_simple_chain(system_prompt="你是一個有用的中文助手。")

# 執行（MessagesState 支援 tuple 格式）
try:
    result = graph.invoke({"messages": [("user", "什麼是 LangGraph？")]})
    print("=== Workflow 結果 ===")
    print(f"最後回應: {result['messages'][-1].content[:200]}")
except Exception as e:
    print(f"[警告] 執行失敗：{e}")
    print("請確認 LLM 服務已啟動。")

## 範例二：自定義多步驟 Workflow

直接使用 LangGraph `StateGraph` API 建構自定義 workflow：

```
preprocess → call_llm → postprocess → END
```

In [ ]:
# 定義自定義 node functions
def preprocess(state: dict) -> dict:
    """前處理：擷取 user message 並加入分析指令。"""
    messages = state.get("messages", [])
    user_text = ""
    for msg in reversed(messages):
        if isinstance(msg, dict) and msg.get("role") == "user":
            user_text = msg["content"]
            break
        elif hasattr(msg, "type") and msg.type == "human":
            user_text = msg.content
            break

    print(f"[preprocess] 擷取輸入: {user_text[:50]}...")

    enhanced_prompt = f"請針對以下內容進行分析並提供摘要：\n\n{user_text}"
    return {
        "messages": [("user", enhanced_prompt)],
    }


def postprocess(state: dict) -> dict:
    """後處理：印出統計資訊。"""
    messages = state.get("messages", [])
    last = messages[-1] if messages else None
    response_text = last.content if hasattr(last, "content") else str(last)
    print(f"[postprocess] 回應長度: {len(response_text)} 字元")
    return {}


print("自定義 node 定義完成。")

In [ ]:
# 使用 LangGraph StateGraph 組裝 workflow
# create_call_llm_node 自動從 llm_config.yaml 取得 ChatLiteLLM
call_llm = create_call_llm_node(system_prompt="你是專業的文件分析助手。")

graph = StateGraph(BaseState)
graph.add_node("preprocess", preprocess)
graph.add_node("call_llm", call_llm)
graph.add_node("postprocess", postprocess)

graph.set_entry_point("preprocess")
graph.add_edge("preprocess", "call_llm")
graph.add_edge("call_llm", "postprocess")
graph.add_edge("postprocess", END)

compiled = graph.compile()
print("Workflow 組裝完成：preprocess → call_llm → postprocess → END")

In [ ]:
# 執行自定義 workflow
try:
    final = compiled.invoke({
        "messages": [("user", "LLM 是一種大型語言模型，能夠理解並生成自然語言。它透過大量文本訓練，學會了語言的統計規律。")],
    })
    print("\n=== Workflow 執行完成 ===")
    print(f"訊息記錄（共 {len(final['messages'])} 則）")
    for msg in final["messages"]:
        role = msg.type if hasattr(msg, "type") else msg.get("role", "?")
        content = msg.content if hasattr(msg, "content") else msg.get("content", "")
        print(f"  [{role}] {content[:80]}...")
except Exception as e:
    print(f"[警告] Workflow 執行失敗：{e}")

## 範例三：帶條件分支的 Workflow

使用 `add_conditional_edges` 根據 state 動態決定下一個節點。

自行擴展 `BaseState` 加入所需欄位：

In [ ]:
from typing import Any


# 擴展 BaseState 加入自定義欄位
class TaskState(BaseState):
    task_type: str = "analyze"
    result_prefix: str = ""


def router(state: dict) -> str:
    """根據 task_type 決定路由。"""
    task = state.get("task_type", "analyze")
    if task == "summarize":
        return "summarize"
    return "analyze"


def summarize_node(state: dict) -> dict:
    print("[summarize] 執行摘要任務")
    return {"result_prefix": "（摘要模式）"}


def analyze_node(state: dict) -> dict:
    print("[analyze] 執行分析任務")
    return {"result_prefix": "（分析模式）"}


# 建構帶條件分支的 workflow
call_llm = create_call_llm_node()

g = StateGraph(TaskState)
g.add_node("call_llm", call_llm)
g.add_node("summarize", summarize_node)
g.add_node("analyze", analyze_node)

g.set_entry_point("call_llm")
g.add_conditional_edges("call_llm", router, {"summarize": "summarize", "analyze": "analyze"})
g.add_edge("summarize", END)
g.add_edge("analyze", END)

conditional_graph = g.compile()

# 測試不同路由
for task in ["summarize", "analyze"]:
    try:
        result = conditional_graph.invoke({
            "messages": [("user", "Hello")],
            "task_type": task,
        })
        print(f"  Task={task} → prefix: {result.get('result_prefix', '')}")
    except Exception as e:
        print(f"  Task={task} → 失敗: {e}")

## 小結

| 元件 | 說明 |
|------|------|
| `BaseState` | 最小狀態（只有 messages），可自行擴展 |
| `create_call_llm_node()` | 自動取得 ChatLiteLLM，建立 LLM 呼叫 node |
| `StateGraph` | LangGraph 原生 API，直接建構 workflow |
| `build_simple_chain()` | 預建的簡單 LLM chain |
| `add_conditional_edges()` | 根據 state 動態路由 |

### MLflow Tracing

- `mlflow.langchain.autolog()` 自動追蹤所有 LangGraph 呼叫
- `mlflow.litellm.autolog()` 自動追蹤所有 LiteLLM 呼叫
- 啟動 `mlflow ui --port 5000` 查看完整 trace

### 後續步驟

- `03_evaluation.ipynb`：使用 MLflow GenAI evaluate 進行評估
- `04_prompt_management.ipynb`：Prompt 註冊、版本管理與自動優化